# Fine-tuning a ranker

Arm 4 is the best arm in `notebooks/04_arms.ipynb`: the fine-tuned model scores each of ~32 rule-generated candidates and keeps the top five. That model was trained to **write** metabolites, and arm 4 reuses its generative likelihood as a ranking score. It was never trained to tell a right candidate from a wrong one.

The gap is measurable. A ranker that orders the same candidate lists at random scores 12.8% recall@5; arm 4 scores 30.7%; the best a top-5 ranker could do over the same lists is 71.5%: the rules reach the answer for 72.2% of pairs, and two parents have six reachable answers against five slots. Arm 4 captures 2.4x chance and leaves **40.8 points** on the table inside lists that already contain the answer. That is larger than any other lever, and it is a ranking problem.

Arm 6 trains the same base model on the ranking decision directly, and is scored against arm 4 on the same test set through the same harness.

| | arm 4 | arm 6 |
|---|---|---|
| trained to | write a metabolite | rank a candidate |
| training example | `<parent>P</parent>` -> `M` | `<parent>P</parent><candidate>C</candidate>` -> `yes` / `no` |
| scored by | mean log-prob per token of the candidate | `logP(yes) - logP(no)` |
| sees wrong answers in training | never | ~8 per right answer, half of them near-misses |

`yes` and `no` are each a single token and neither merges with the prompt, so the score needs no length normalisation and cannot be distorted by candidate length. The arm-4 scorer divides by token count to correct for that.

## 1. Building the training data

In [1]:
import datetime as dt
import json, os, pathlib, random, re, sys, time
import numpy as np, pandas as pd

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from src import evaluation as ev
from src import decoding as dec

for line in (ROOT / ".env").read_text().splitlines():
    if "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()

import tinker
import tinker.types as tt

BASE_MODEL = "Qwen/Qwen3-8B"
TODAY = dt.date.today().isoformat()

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

rules = pd.read_csv(ROOT / "rules" / "metabolism_rules.csv")
COMPILED = []
for _, r in rules.iterrows():
    rxn = AllChem.ReactionFromSmarts(r["smirks"]); rxn.Initialize()
    COMPILED.append((rxn, bool(r["keep_largest_fragment"])))

def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    return max(frags, key=lambda f: f.GetNumHeavyAtoms()) if len(frags) > 1 else mol

def enumerate_candidates(parent, per_rule_cap=200):
    """The same enumeration notebook 04 uses, so arm 6 ranks the same lists as arms 2 and 4."""
    mol = Chem.MolFromSmiles(parent)
    if mol is None: return []
    out = {}
    for rxn, keep in COMPILED:
        try: products = rxn.RunReactants((mol,))
        except Exception: continue
        n = 0
        for tup in products:
            for p in tup:
                try:
                    if keep: p = largest_fragment(p)
                    Chem.SanitizeMol(p); out[Chem.MolToSmiles(p, canonical=True)] = 1
                except Exception: pass
                n += 1
                if n > per_rule_cap: break
            if n > per_rule_cap: break
    return list(out)

print(f"{len(COMPILED)} rules compiled")

/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_seq_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_path" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_owner" has conflict with protected namespace "model_".

You may be able to res

42 rules compiled


One example per (parent, candidate) pair: the candidate is a product of the 42 rules applied to that parent, and the label is whether it matches a recorded metabolite of that parent under the same standardised key the scorer uses. The negatives are the exact distractors the model faces at test time, and half of each parent's negatives are its closest wrong candidates by Morgan fingerprint, so it has to separate a real metabolite from a near-miss.

**`TRAINING_MIX` selects the data.** `"base"` is DrugBank + GLORYx + MetXBioDB, the split prepared in notebook 02. `"base+gostar"` adds GOSTAR's **mammalian** records.

GOSTAR is filtered before it is added. Of its 18,093 rows only 25.6% are human and **23.2% are *Salmonella typhimurium***: Ames-test records, with `Nitroreductase` and `Azoreductase` in the enzyme column. That is bacterial mutagenicity, and training on it would teach the model the wrong chemistry under the right label. The species column also carries compound values (`Human; Rat`, `Mouse; Rat`, `Mammal`) across 260 distinct strings, so the filter parses them.

Two separations are enforced:

- **GOSTAR parents that appear in the holdout test set are dropped.** One does.
- **GOSTAR parents that appear in the development set are dropped too**, so the dev split is byte-identical between the two mixes and only the training data differs.

In [2]:
import re
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

TRAINING_MIX = "base+gostar"      # "base" or "base+gostar"
NEG_PER_POS = 8

_fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def _fp(s):
    m = Chem.MolFromSmiles(s)
    return _fpgen.GetFingerprint(m) if m else None

base_train = [json.loads(l) for l in (ROOT / "data/processed/train.jsonl").open()]
base_dev   = [json.loads(l) for l in (ROOT / "data/processed/dev.jsonl").open()]
def group(rows):
    out = {}
    for r in rows: out.setdefault(r["parent"], set()).add(r["target"])
    return out
train_by_parent, dev_by_parent_pairs = group(base_train), group(base_dev)

if TRAINING_MIX == "base+gostar":
    MAMMAL = {"human","rat","mouse","dog","monkey","rabbit","guinea pig","hamster","pig","cat","mammal"}
    def is_mammal(s):
        parts = [p.strip().lower() for p in re.split(r"[;,/]", s or "") if p.strip()]
        return bool(parts) and all(p in MAMMAL for p in parts)

    gostar = pd.read_csv(ROOT / "data/gostar.csv", dtype=str, keep_default_na=False, na_values=[])
    mam = gostar[gostar["species"].map(is_mammal)]
    test_raw = pd.read_csv(ROOT / "data/test_set.csv", dtype=str, keep_default_na=False, na_values=[])
    test_keys = {ev.standardize(s) for s in test_raw["parent_smiles"]}
    dev_keys  = {ev.standardize(p) for p in dev_by_parent_pairs}

    added = dropped_test = dropped_dev = 0
    for a, b in zip(mam["parent_smiles"], mam["metabolite_smiles"]):
        p, t = ev.canonical(a), ev.canonical(b)
        if p is None or t is None: continue
        k = ev.standardize(p)
        if k in test_keys:  dropped_test += 1; continue
        if k in dev_keys:   dropped_dev += 1; continue
        train_by_parent.setdefault(p, set()).add(t); added += 1

    print(f"  GOSTAR: {len(gostar):,} rows -> {len(mam):,} mammalian")
    print(f"    dropped, parent is in the test set : {dropped_test}")
    print(f"    dropped, parent is in dev          : {dropped_dev}")
    print(f"    added to training                  : {added:,}")

print(f"\n  {TRAINING_MIX}: {len(train_by_parent):,} training parents, "
      f"{sum(len(v) for v in train_by_parent.values()):,} pairs")
print(f"  dev is unchanged: {len(dev_by_parent_pairs)} parents, "
      f"{sum(len(v) for v in dev_by_parent_pairs.values())} pairs")


def build_examples(by_parent, label, rng_seed=0):
    """Enumerate each parent's rule candidates, label them, and sample negatives."""
    rng = random.Random(rng_seed)
    out, n_pos, n_neg, reach, total = [], 0, 0, 0, 0
    t0 = time.time()
    for i, (parent, targets) in enumerate(sorted(by_parent.items()), 1):
        cands = enumerate_candidates(parent)
        total += len(targets)
        if not cands: continue
        ckeys = [ev.match_key(c, "standardized") for c in cands]   # once per candidate; this is
        tkeys = {ev.match_key(t, "standardized") for t in targets} # the expensive call
        reach += len(tkeys & set(ckeys))
        pos = [c for c, k in zip(cands, ckeys) if k in tkeys]
        neg = [c for c, k in zip(cands, ckeys) if k not in tkeys]
        if not pos or not neg: continue
        pfps = [f for f in (_fp(p) for p in pos) if f]
        scored = []
        for c in neg:
            cf = _fp(c)
            s = max((DataStructs.TanimotoSimilarity(cf, pf) for pf in pfps), default=0.0) if cf else 0.0
            scored.append((s, c))
        scored.sort(reverse=True, key=lambda x: x[0])
        k = NEG_PER_POS * len(pos)
        hard = [c for _, c in scored[:k//2]]
        rest = [c for _, c in scored[k//2:]]
        easy = rng.sample(rest, min(k - len(hard), len(rest))) if rest else []
        for c in pos: out.append({"parent": parent, "candidate": c, "label": "yes"}); n_pos += 1
        for c in hard + easy: out.append({"parent": parent, "candidate": c, "label": "no"}); n_neg += 1
        if i % 500 == 0:
            print(f"    {label}: {i}/{len(by_parent)} parents  ({time.time()-t0:.0f}s)", flush=True)
    rng.shuffle(out)
    print(f"  {label}: {len(out):,} examples ({n_pos:,} yes / {n_neg:,} no) over "
          f"{len({r['parent'] for r in out})} parents, rule coverage {reach/max(total,1):.1%} "
          f"[{time.time()-t0:.0f}s]")
    return out

print()
train_rows = build_examples(train_by_parent, "train", 0)
dev_rows   = build_examples(dev_by_parent_pairs, "dev", 1)
for name, rows in (("rerank_train", train_rows), ("rerank_dev", dev_rows)):
    (ROOT / f"data/processed/{name}.jsonl").write_text("\n".join(json.dumps(r) for r in rows))
frac = lambda rs: sum(r["label"] == "yes" for r in rs) / len(rs)
print(f"\n  train {len(train_rows):,} examples, {frac(train_rows):.1%} positive")
print(f"  dev   {len(dev_rows):,} examples, {frac(dev_rows):.1%} positive")

  GOSTAR: 18,093 rows -> 11,687 mammalian
    dropped, parent is in the test set : 1
    dropped, parent is in dev          : 244
    added to training                  : 11,442

  base+gostar: 8,507 training parents, 15,286 pairs
  dev is unchanged: 250 parents, 437 pairs



    train: 1500/8507 parents  (907s)


    train: 3000/8507 parents  (1699s)


    train: 3500/8507 parents  (1856s)


    train: 4500/8507 parents  (2487s)


...

    train: 6000/8507 parents  (3410s)


    train: 7500/8507 parents  (4109s)


  train: 60,520 examples (7,324 yes / 53,196 no) over 4718 parents, rule coverage 47.1% [4282s]


  dev: 2,536 examples (299 yes / 2,237 no) over 201 parents, rule coverage 68.2% [139s]



  train 60,520 examples, 12.1% positive
  dev   2,536 examples, 11.8% positive


**What a `no` means here.** It means *not recorded as a metabolite of this parent in the source tables*. Resveratrol paired with dihydro-resveratrol, a real human gut metabolite, is labelled `no` because resveratrol's rows do not list it. The model therefore trains on positive-unlabelled data, and some of its negatives are genuine metabolites nobody wrote down.

This makes training *harder*, since the model is penalised for correct chemistry, so any gain measured here is a lower bound. The evaluation makes the same assumption: `precision_known` in `src/evaluation.py` is documented as a lower bound for the same reason. The one risk is that a ranker learns the source tables' coverage biases, which is why arm 6 is scored on held-out test molecules and against an untuned control.

In [3]:
r = train_rows[0]
print("one training example, exactly as the model receives it:")
print(f"  shown : <parent>{r['parent'][:30]}...</parent>")
print(f"          <candidate>{r['candidate'][:30]}...</candidate>")
print(f"  writes: {r['label']!r}")

one training example, exactly as the model receives it:
  shown : <parent>CC(C=CC=C(C)C=CC1=C(C)CCCC1(C)...</parent>
          <candidate>CC(C=CC=C(C)C=CC1=C(C)CCCC1(C)...</candidate>
  writes: 'no'


In [4]:
PROMPT = "<parent>{parent}</parent>\n<candidate>{candidate}</candidate>\n"

service = tinker.ServiceClient(api_key=os.environ["TINKER_API_KEY"])
training = service.create_lora_training_client(
    base_model=BASE_MODEL, rank=32, train_mlp=True, train_attn=True, train_unembed=True)
tokenizer = training.get_tokenizer()
EOS = tokenizer.eos_token_id or 0

YES, NO = tokenizer.encode("yes"), tokenizer.encode("no")
assert len(YES) == len(NO) == 1, "the verdict must be one token for the score to need no normalising"
_p = PROMPT.format(parent="CCO", candidate="CCOC")
assert tokenizer.encode(_p + "yes")[:len(tokenizer.encode(_p))] == tokenizer.encode(_p), \
    "the prompt's last token merges with the verdict; scoring would be misaligned"
print(f"verdict tokens: yes={YES[0]}  no={NO[0]}, single-token and no merge with the prompt")

def to_datum(row):
    """Prompt is context; the one-token verdict and the EOS after it are the supervised targets."""
    prompt_ids = tokenizer.encode(PROMPT.format(**row))
    answer_ids = tokenizer.encode(row["label"])
    ids = prompt_ids + answer_ids
    return tt.Datum(
        model_input=tt.ModelInput.from_ints(ids),
        loss_fn_inputs={"target_tokens": ids[1:] + [EOS],
                        "weights": [0.0] * (len(prompt_ids) - 1) + [1.0] * len(answer_ids) + [1.0]})

def n_supervised(rows):
    return sum(len(tokenizer.encode(r["label"])) + 1 for r in rows)

def batch_loss(fb_output, supervised):
    total = sum(float(np.asarray(o["elementwise_loss"].data).sum())
                for o in fb_output.loss_fn_outputs)
    return total / max(supervised, 1)

_el = np.asarray(training.forward([to_datum(train_rows[0])], "cross_entropy").result()
                 .loss_fn_outputs[0]["elementwise_loss"].data)
_n_prompt = len(tokenizer.encode(PROMPT.format(**train_rows[0]))) - 1
assert _el[:_n_prompt].sum() == 0, "prompt positions carry loss; the weights are wrong"
assert _el[_n_prompt] != 0, "the verdict position carries no loss; the mask is one token late"
lengths = [len(tokenizer.encode(PROMPT.format(**r))) for r in train_rows[:500]]
print(f"masking verified. ~{np.mean(lengths):.0f} tokens per example, "
      f"~{np.mean(lengths)*len(train_rows)/1e6:.2f}M tokens per epoch")

verdict tokens: yes=9693  no=2152, single-token and no merge with the prompt


masking verified. ~77 tokens per example, ~4.67M tokens per epoch


## 2. How the ranker is scored, and how checkpoints are chosen

Checkpoints are selected on **dev recall@5**, the headline metric. In notebook 03, dev loss bottomed out in epoch 0 and then rose for the rest of the run while recall kept climbing, so selecting on that proxy would have picked a much earlier checkpoint. Yes/no accuracy is the same kind of proxy.

The probe ranks the **full** candidate list for a sample of dev molecules, the way arm 6 ranks test molecules: every candidate scored, top five kept, scored by `src/evaluation.py`.

In [5]:
K = 5
PROBE_PARENTS = 40

dev_pairs = [json.loads(l) for l in (ROOT / "data/processed/dev.jsonl").open()]
dev_by_parent = {}
for r in dev_pairs: dev_by_parent.setdefault(r["parent"], set()).add(r["target"])
rng = np.random.default_rng(0)
probe_parents = sorted(rng.choice(sorted(dev_by_parent), size=PROBE_PARENTS, replace=False))
probe_cands = {p: enumerate_candidates(p) for p in probe_parents}
n_calls = sum(len(v) for v in probe_cands.values()) * 2
print(f"probe: {PROBE_PARENTS} dev molecules, "
      f"{sum(len(v) for v in probe_cands.values())} candidates, {n_calls} scoring calls per evaluation")


def score_candidates(client, parent, cands):
    """logP(yes) - logP(no) for each candidate, under one model.

    The log-odds rather than logP(yes) alone: both verdicts are conditioned on the same prompt,
    so the difference cancels whatever that particular prompt costs in absolute probability and
    leaves only the model's preference between the two answers. Prefix and verdict are encoded
    separately and concatenated, so the token boundary cannot shift with the candidate.
    """
    futures = []
    for c in cands:
        prefix = tokenizer.encode(PROMPT.format(parent=parent, candidate=c))
        futures.append(tuple(
            client.compute_logprobs(prompt=tt.ModelInput.from_ints(prefix + v)) for v in (YES, NO)))
    out = []
    for f_yes, f_no in futures:
        out.append(float(f_yes.result()[-1]) - float(f_no.result()[-1]))
    return out


def probe_recall(client, k=K):
    preds = {}
    for p, cands in probe_cands.items():
        preds[p] = (dec.rank_candidates(cands, score_candidates(client, p, cands), p, k=k).predictions
                    if cands else [])
    pairs = [(p, t) for p in probe_parents for t in dev_by_parent[p]]
    df = ev.evaluate([p for p, _ in pairs], [t for _, t in pairs],
                     [preds[p] for p, _ in pairs], k_values=(1, k), criterion="standardized")
    return float(df.loc[k, "recall_pair"]), float(df.loc[1, "recall_pair"])

untuned_r5, untuned_r1 = probe_recall(service.create_sampling_client(base_model=BASE_MODEL))
print(f"untuned model on this scoring format: recall@5 {untuned_r5:.1%}  recall@1 {untuned_r1:.1%}")

probe: 40 dev molecules, 1368 candidates, 2736 scoring calls per evaluation


untuned model on this scoring format: recall@5 20.6%  recall@1 4.4%


## 3. Smoke run

In [6]:
BATCH = 48
LR = 1e-4
SMOKE_STEPS = 20

random.seed(0)
shuffled = train_rows[:]
random.shuffle(shuffled)

losses = []
t0 = time.time()
for step in range(SMOKE_STEPS):
    rows = shuffled[step*BATCH:(step+1)*BATCH]
    fb = training.forward_backward([to_datum(r) for r in rows], "cross_entropy")
    training.optim_step(adam_params=tinker.AdamParams(learning_rate=LR)).result()
    losses.append(batch_loss(fb.result(), n_supervised(rows)))
    if step % 5 == 0: print(f"  step {step:>2}  loss {losses[-1]:.4f}")
secs_per_step = (time.time() - t0) / SMOKE_STEPS
first, last = np.mean(losses[:3]), np.mean(losses[-3:])
print(f"\n  loss {first:.4f} -> {last:.4f}  ({secs_per_step:.1f}s/step)")
assert last < first, f"loss did not fall ({first:.4f} -> {last:.4f}); fix that before section 4"
print("  gate passed")

  step  0  loss 16.5246


  step  5  loss 8.8662


  step 10  loss 1.6162


  step 15  loss 0.2903



  loss 15.1575 -> 0.2283  (2.9s/step)
  gate passed


## 4. Training

In [7]:
EPOCHS = 4
CHECKPOINT_TTL = 7 * 24 * 3600
steps_per_epoch = len(shuffled) // BATCH
EVAL_EVERY = max(1, steps_per_epoch // 2)

print(f"  {steps_per_epoch} steps per epoch, {steps_per_epoch*EPOCHS} total, "
      f"~{steps_per_epoch*EPOCHS*secs_per_step/60:.0f} min of training")
print(f"  dev probe every {EVAL_EVERY} steps\n")

history = []
best = {"recall": -1.0, "step": -1, "path": None}
step = 0
t0 = time.time()

def measure():
    path = training.save_weights_for_sampler(
        name=f"rank{step}", ttl_seconds=CHECKPOINT_TTL).result().path
    r5, r1 = probe_recall(service.create_sampling_client(model_path=path))
    return path, r5, r1

for epoch in range(EPOCHS):
    random.shuffle(shuffled)
    for i in range(0, len(shuffled) - BATCH + 1, BATCH):
        rows = shuffled[i:i+BATCH]
        fb = training.forward_backward([to_datum(r) for r in rows], "cross_entropy")
        training.optim_step(adam_params=tinker.AdamParams(learning_rate=LR)).result()
        row = {"step": step, "epoch": epoch, "loss": batch_loss(fb.result(), n_supervised(rows)),
               "dev_recall5": None, "dev_recall1": None, "path": None}
        if step % EVAL_EVERY == 0:
            path, r5, r1 = measure()
            row.update({"dev_recall5": r5, "dev_recall1": r1, "path": path})
            flag = ""
            if r5 > best["recall"]:
                best = {"recall": r5, "step": step, "path": path}; flag = "  <- best so far"
            print(f"  step {step:>4} ep {epoch}  loss {row['loss']:.4f}  "
                  f"dev@5 {r5:>5.1%}  dev@1 {r1:>5.1%}{flag}")
        history.append(row)
        step += 1

if history[-1]["dev_recall5"] is None:
    step -= 1
    path, r5, r1 = measure()
    history[-1].update({"dev_recall5": r5, "dev_recall1": r1, "path": path})
    if r5 > best["recall"]: best = {"recall": r5, "step": step, "path": path}
    print(f"  step {step:>4} (final)  dev@5 {r5:>5.1%}  dev@1 {r1:>5.1%}")
    step += 1

hist = pd.DataFrame(history)
assert hist["dev_recall5"].notna().sum() >= 2, "fewer than two dev evaluations landed"
# Named after the mix so both records survive; notebook 06 reads reranker_base_* and reranker_*.
PREFIX = "reranker_base" if TRAINING_MIX == "base" else "reranker"
hist.to_csv(ROOT / f"data/processed/{PREFIX}_history.csv", index=False)
print(f"\n  {step} steps in {(time.time()-t0)/60:.1f} min")
print(f"  best dev recall@5 {best['recall']:.1%} at step {best['step']}  (untuned {untuned_r5:.1%})")

(ROOT / f"data/processed/{PREFIX}_run.json").write_text(json.dumps({
    "base_model": BASE_MODEL, "training_mix": TRAINING_MIX,
    "rank": 32, "batch": BATCH, "learning_rate": LR, "epochs": EPOCHS,
    "training_steps": step, "smoke_steps": SMOKE_STEPS, "train_examples": len(train_rows),
    "best_step": best["step"], "best_checkpoint": best["path"],
    "checkpoint_ttl_seconds": CHECKPOINT_TTL, "saved_on": TODAY,
    "probe_molecules": PROBE_PARENTS, "probe_recall5": best["recall"],
    "untuned_probe_recall5": untuned_r5, "untuned_probe_recall1": untuned_r1,
    "all_checkpoints": {int(r.step): r.path for r in hist.dropna(subset=["path"]).itertuples()},
}, indent=2))
print(f"  recorded -> data/processed/{PREFIX}_run.json")


  1260 steps per epoch, 5040 total, ~246 min of training
  dev probe every 630 steps



  step    0 ep 0  loss 0.3103  dev@5 26.5%  dev@1  4.4%  <- best so far


  step  630 ep 0  loss 0.1082  dev@5 44.1%  dev@1 11.8%  <- best so far


  step 1260 ep 1  loss 0.0821  dev@5 47.1%  dev@1 14.7%  <- best so far


  step 1890 ep 1  loss 0.1668  dev@5 48.5%  dev@1 14.7%  <- best so far


...

  step 3780 ep 3  loss 0.0592  dev@5 47.1%  dev@1 11.8%


  step 4410 ep 3  loss 0.0807  dev@5 42.6%  dev@1 13.2%


  step 5039 (final)  dev@5 45.6%  dev@1 16.2%



  5040 steps in 283.5 min
  best dev recall@5 48.5% at step 1890  (untuned 20.6%)


  recorded -> data/processed/reranker_run.json


## 5. Arm 6: Fine-tuned to rank, on the holdout test set

The ranker trained above scores every rule candidate yes/no and the top five are kept.

In [8]:
test = pd.read_csv(ROOT / "data" / "test_set.csv", dtype=str, keep_default_na=False, na_values=[])
test["parent"] = [ev.canonical(s) for s in test["parent_smiles"]]
test["metabolite"] = [ev.canonical(s) for s in test["metabolite_smiles"]]
parents, references = list(test["parent"]), list(test["metabolite"])
unique_parents = sorted(set(parents))
candidates = {p: enumerate_candidates(p) for p in unique_parents}
print(f"test set: {len(test)} pairs over {len(unique_parents)} molecules, "
      f"{sum(len(v) for v in candidates.values())} candidates to score")

ranker = service.create_sampling_client(model_path=best["path"])

def run_ranking_arm(client, label, report_every=40):
    t0 = time.time(); out = {}
    for i, p in enumerate(unique_parents, 1):
        cands = candidates[p]
        if not cands:
            out[p] = dec.Decoded(predictions=[]); continue
        out[p] = dec.rank_candidates(cands, score_candidates(client, p, cands), p, k=K)
        if i % report_every == 0:
            print(f"  {label}: {i}/{len(unique_parents)}  ({time.time()-t0:.0f}s)")
    print(f"  {label}: done in {time.time()-t0:.0f}s")
    return out

arm6 = run_ranking_arm(ranker, "arm 6")
arm6_untuned = run_ranking_arm(service.create_sampling_client(base_model=BASE_MODEL), "control")

test set: 309 pairs over 155 molecules, 5243 candidates to score


  arm 6: 40/155  (286s)


  arm 6: 80/155  (578s)


  arm 6: 120/155  (853s)


  arm 6: done in 1122s


  control: 40/155  (68s)


  control: 80/155  (172s)


  control: 120/155  (236s)


  control: done in 306s


The control is the same yes/no scoring format with the **untuned** model. Without it, arm 6 against arm 4 would confound a different training objective with a different scoring format. Arm 6 minus this control is what the ranking fine-tune is worth; the control against arm 2 is what the format alone is worth.

In [9]:
def score_arm(decoded, label):
    df = ev.evaluate(parents, references, [decoded[p].predictions for p in parents],
                     k_values=(1, 3, 5), criterion="standardized")
    df.insert(0, "arm", label); return df

SUFFIX = "base mix" if TRAINING_MIX == "base" else "+GOSTAR mammalian"
LAB_ARM, LAB_CTL = f"6  tuned ranker ({SUFFIX})", f"6c untuned control ({SUFFIX} run)"

# Replace this mix's rows instead of appending, or at5[label] returns a Series.
prev = pd.read_csv(ROOT / "docs" / "arm_results.csv", index_col=0)
prev = prev[~prev["arm"].isin([LAB_ARM, LAB_CTL])]
results = pd.concat([prev, score_arm(arm6_untuned, LAB_CTL), score_arm(arm6, LAB_ARM)])
assert not results.reset_index().duplicated(["k", "arm"]).any(), "duplicate (k, arm) rows"
results.to_csv(ROOT / "docs" / "arm_results.csv")

at5 = results.loc[5].set_index("arm")["recall_pair"]
at1 = results.loc[1].set_index("arm")["recall_pair"]
A4, A6 = "4  tuned, ranking", LAB_ARM
print(f"  arm 4, generator reused as a ranker : recall@5 {at5[A4]:>6.1%}   recall@1 {at1[A4]:>6.1%}")
print(f"  arm 6, trained to rank              : recall@5 {at5[A6]:>6.1%}   recall@1 {at1[A6]:>6.1%}")
print(f"  difference                          : {(at5[A6]-at5[A4])*100:>+6.1f}pp  {(at1[A6]-at1[A4])*100:>+6.1f}pp")
print(f"\n  perfect ranker over the same lists  : {sum(1 for p, r in zip(parents, references) if ev.match_key(r,'standardized') in {ev.match_key(c,'standardized') for c in candidates[p]})/len(parents):.1%}")
results[results.index == 5].round(3)


  arm 4, generator reused as a ranker : recall@5  30.7%   recall@1  12.9%
  arm 6, trained to rank              : recall@5  46.6%   recall@1  18.1%
  difference                          :  +15.9pp    +5.2pp

  perfect ranker over the same lists  : 72.2%


,arm,recall_pair,ceiling,pct_of_ceiling,recall_parent,parent_hit_rate,precision_known,tanimoto_wrong,parent_echo_rate
k,,,,,,,,,
5,"1 untuned, direct (few-shot)",0.049,0.964,0.050,0.055,0.084,0.018,0.534,0.0
5,"1b untuned, direct (bare)",0.013,0.964,0.013,0.026,0.026,0.005,0.339,0.0
5,"2 untuned, ranking",0.204,0.964,0.211,0.198,0.342,0.080,0.577,0.0
5,"3 tuned, direct",0.214,0.964,0.221,0.244,0.368,0.085,0.512,0.0
5,"4 tuned, ranking",0.307,0.964,0.319,0.291,0.458,0.120,0.578,0.0
5,"5 tuned, pooled",0.223,0.964,0.232,0.229,0.348,0.089,0.584,0.0
5,6c untuned control (base-mix run),0.162,0.964,0.168,0.151,0.265,0.063,0.567,0.0
5,6 tuned ranker (base mix),0.460,0.964,0.477,0.466,0.613,0.179,0.580,0.0
5,"6c untuned, yes/no scoring",0.162,0.964,0.168,0.151,0.265,0.063,0.567,0.0


### Against published figures

MetaTrans counts a prediction correct when its Morgan-fingerprint Tanimoto to a reference is
exactly 1.0. That rule discards stereochemistry and atom charge but not tautomers, while the
standardised-SMILES matching used above merges tautomers and charge states that fingerprints
separate. Neither rule contains the other, so the two sets of numbers are not comparable until one
rule is applied to both.

Applying their rule to this arm's own top five settles the matching difference. It does not settle
the test set: every row of `data/test_set.csv` carries `source = MetaTrans`, but the provided data
is modelled after theirs rather than identical to it, so this is a close comparison and not a
head-to-head.

Published figures are the top-5 window of Table 4 in Litsa, Das & Kavraki, *Chem. Sci.* 2020, held
in `docs/published_comparison.csv` next to the row this cell computes. `compute_logprobs` is not
bit-reproducible, so a rerun can move the computed row by about one pair.

In [ ]:
PUBLISHED = ROOT / "docs" / "published_comparison.csv"
_FP = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def fingerprint_equal(a, b):
    """MetaTrans's match rule: Morgan-fingerprint Tanimoto of exactly 1.0, molecules as written.

    Not standardised first: their rule drops stereochemistry and charge because this fingerprint
    is chirality-blind, and leaves tautomers separate.
    """
    ma, mb = Chem.MolFromSmiles(a), Chem.MolFromSmiles(b)
    if ma is None or mb is None:
        return False
    return DataStructs.TanimotoSimilarity(_FP.GetFingerprint(ma), _FP.GetFingerprint(mb)) == 1.0

truths = {}
for p, r in zip(parents, references):
    truths.setdefault(p, []).append(r)
top5 = {p: arm6[p].predictions[:K] for p in unique_parents}

hits = sum(any(fingerprint_equal(q, r) for q in top5[p]) for p, r in zip(parents, references))
any_hit = [any(fingerprint_equal(q, r) for q in top5[p] for r in rs) for p, rs in truths.items()]
precision = [sum(any(fingerprint_equal(q, r) for r in rs) for q in top5[p]) / K
             for p, rs in truths.items()]

label = f"this pipeline ({SUFFIX} ranker)"
table = pd.read_csv(PUBLISHED, index_col="system")
table.loc[label, ["recall@5", "precision@5", "any_hit@5", "matching", "source"]] = [
    hits / len(references), float(np.mean(precision)), float(np.mean(any_hit)),
    "fingerprint equality", "notebooks/05_reranker.ipynb"]
table.to_csv(PUBLISHED)

print(f"{label}: recall@5 {hits/len(references):.1%}   precision@5 {np.mean(precision):.1%}   "
      f"drugs with >=1 hit {np.mean(any_hit):.1%}")
table.round(3)

## 6. Every arm, side by side

Six of the nine arms are plotted: no training, training to write, the writer reused to rank, training to rank. The three ablations stay in `docs/arm_results.csv`, which the cell below reads in full.

The dashed line is the ceiling for the **ranking** arms only: they can return nothing the rules did not produce, and nothing that does not fit one of the five slots. That is 71.5%, below the rules' 72.2% coverage, because two parents have six reachable answers and five slots. The generating arms are not bound by it and could in principle exceed it; in practice none comes close.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ACCENT, INK, MUTED, GRID, SURFACE = "#eb6834", "#0b0b0b", "#6b6a66", "#e2e1dd", "#fcfcfb"
HOLLOW = "#9a9892"
at5 = pd.read_csv(ROOT / "docs" / "arm_results.csv", index_col=0).loc[5].set_index("arm")["recall_pair"]

# Three ways of using the model, each shown untuned and fine-tuned. The names are the ones used
# everywhere else in this project.
ROWS = [("Generative",                      "1  untuned, direct (few-shot)", "3  tuned, direct"),
        ("Generative used as ranker",       "2  untuned, ranking",           "4  tuned, ranking"),
        ("Ranking, yes/no",                 "6c untuned control (base mix run)", "6  tuned ranker (base mix)")]

# A perfect top-5 ranker, and a random ordering of the same lists, bound every ranking row.
reach = {}
for p_, r_ in zip(parents, references):
    key = ev.match_key(r_, "standardized")
    if key is not None and key in {ev.match_key(c, "standardized") for c in candidates[p_]}:
        d = reach.setdefault(p_, {}); d[key] = d.get(key, 0) + 1
cap = sum(sum(sorted(d.values(), reverse=True)[:K]) for d in reach.values()) / len(parents)
rand = sum(min(K / max(len(candidates[p_]), 1), 1.0)
           for p_, r_ in zip(parents, references)
           if ev.match_key(r_, "standardized") in
              {ev.match_key(c, "standardized") for c in candidates[p_]}) / len(parents)

fig, ax = plt.subplots(figsize=(10.5, 4.4))
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
ys = list(range(len(ROWS)))[::-1]
for y, (label, untuned_key, tuned_key) in zip(ys, ROWS):
    u, t = at5[untuned_key], at5[tuned_key]
    ax.annotate("", xy=(t, y), xytext=(u, y),
                arrowprops=dict(arrowstyle="-|>", color=GRID, linewidth=7, shrinkA=9, shrinkB=11))
    ax.scatter([u], [y], s=130, facecolor=SURFACE, edgecolor=HOLLOW, linewidth=2.2, zorder=4)
    ax.scatter([t], [y], s=190, facecolor=ACCENT, edgecolor=SURFACE, linewidth=1.6, zorder=5)
    ax.annotate(f"{u:.1%}", xy=(u, y), xytext=(0, -20), textcoords="offset points",
                ha="center", color=MUTED, fontsize=10.5)
    ax.annotate(f"{t:.1%}", xy=(t, y), xytext=(14, 0), textcoords="offset points",
                va="center", color=INK, fontsize=13, fontweight="bold")

ax.axvline(cap, color=MUTED, linewidth=1.2, linestyle=(0, (5, 3)))
ax.annotate(f"best possible, top five  {cap:.1%}", xy=(cap, len(ROWS) - 0.42),
            xytext=(-7, 0), textcoords="offset points", ha="right", color=MUTED, fontsize=9.5)
ax.axvline(rand, color=MUTED, linewidth=1.0, linestyle=(0, (2, 3)))
ax.annotate(f"random order  {rand:.1%}", xy=(rand, -0.55), xytext=(6, 0),
            textcoords="offset points", ha="left", color=MUTED, fontsize=9.5)

ax.set_yticks(ys); ax.set_yticklabels([r[0] for r in ROWS], fontsize=13)
ax.set_ylim(-0.75, len(ROWS) - 0.15)
ax.set_xlim(0, cap * 1.1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=10.5)
ax.tick_params(axis="y", length=0, colors=INK)
ax.set_xlabel("recall@5 on the 309 holdout pairs", color=MUTED, fontsize=10.5)
ax.set_title("Recall@5 by approach: untuned against fine-tuned",
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=16)
ax.legend(handles=[plt.Line2D([], [], marker="o", linestyle="", markersize=9,
                              markerfacecolor=SURFACE, markeredgecolor=HOLLOW, label="untuned"),
                   plt.Line2D([], [], marker="o", linestyle="", markersize=10,
                              markerfacecolor=ACCENT, markeredgecolor=SURFACE, label="fine-tuned")],
          # upper left is the only empty corner: the ceiling line crosses the lower right
          frameon=False, fontsize=10.5, labelcolor=MUTED, loc="upper left", ncol=2)
plt.savefig(ROOT / "docs" / "arm_comparison.png", dpi=160, bbox_inches="tight", facecolor=SURFACE)
plt.show()